In [1]:
# ETAPA 4 - ANÁLISE DE DADOS
# PARTE 1 - ANÁLISE ESTATÍSTICA DESCRITIVA
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

# Configurações de visualização
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:,.4f}".format)

df = pd.read_csv('dados_empresa_tratado.csv')

# Configuração visual dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10


# ============================================================
# CARREGAMENTO DA BASE TRATADA
# ============================================================

caminho_base = Path("dados_empresa_tratado.csv")

if not caminho_base.exists():
    raise FileNotFoundError(
        "O arquivo 'dados_empresa_tratado.csv' não foi encontrado. "
        "Verifique se ele está na mesma pasta deste notebook."
    )

df = pd.read_csv(caminho_base)

# Converter coluna data para datetime
df["data"] = pd.to_datetime(df["data"], errors="coerce")

print(df.to_string())

     id_venda       data  mes  trimestre      cliente tipo_cliente  tempo_relacionamento_meses faixa_renda segmento     produto    categoria  marca linha_produto  quantidade  preco_unitario  desconto     receita  custo_unitario       lucro        regiao          cidade canal_venda     vendedor  meta_venda atingiu_meta  prazo_entrega_dias status_entrega
0           1 2024-12-14   12          4  Cliente_150         Novo                           1       Média   Varejo       Mouse  Periféricos   Beta        Padrão           5      4,697.1200    0.1800 19,258.1920      3,172.2600  3,396.8920         Norte          Manaus      Online  Vendedor_02 24,931.1100          Não                   9       Atrasado
1           2 2024-03-09    3          1  Cliente_109         Novo                           0        Alta   Varejo  Smartphone    Telefonia  Delta        Padrão          10        506.5400    0.1200  4,457.5520        307.6600  1,380.9520         Norte          Manaus      Online  Vendedo

In [2]:
# VALIDAÇÃO INICIAL DA BASE
# ============================================================

print(f"Quantidade de linhas: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")
print(f"Total de valores nulos: {df.isna().sum().sum()}")
print(f"Total de linhas duplicadas: {df.duplicated().sum()}")

df.info()

Quantidade de linhas: 1000
Quantidade de colunas: 27
Total de valores nulos: 0
Total de linhas duplicadas: 0
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 27 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   id_venda                    1000 non-null   int64         
 1   data                        1000 non-null   datetime64[us]
 2   mes                         1000 non-null   int64         
 3   trimestre                   1000 non-null   int64         
 4   cliente                     1000 non-null   str           
 5   tipo_cliente                1000 non-null   str           
 6   tempo_relacionamento_meses  1000 non-null   int64         
 7   faixa_renda                 1000 non-null   str           
 8   segmento                    1000 non-null   str           
 9   produto                     1000 non-null   str           
 10  categoria              

In [3]:
# 4.1 - KPIs GERAIS
# ============================================================

receita_total = df["receita"].sum()
lucro_total = df["lucro"].sum()
margem_lucro = lucro_total / receita_total
quantidade_total = df["quantidade"].sum()
ticket_medio = df["receita"].mean()
desconto_medio = df["desconto"].mean()
meta_total = df["meta_venda"].sum()
taxa_atingimento_meta = (df["atingiu_meta"] == "Sim").mean()
taxa_atraso = (df["status_entrega"] == "Atrasado").mean()
total_vendas = df["id_venda"].count()

kpis_gerais = pd.DataFrame({
    "KPI": [
        "Total de Vendas",
        "Receita Total",
        "Lucro Total",
        "Margem de Lucro",
        "Quantidade Vendida",
        "Ticket Médio por Venda",
        "Desconto Médio",
        "Meta Total",
        "Taxa de Atingimento de Meta",
        "Taxa de Entregas Atrasadas"
    ],
    "Valor": [
        total_vendas,
        receita_total,
        lucro_total,
        margem_lucro,
        quantidade_total,
        ticket_medio,
        desconto_medio,
        meta_total,
        taxa_atingimento_meta,
        taxa_atraso
    ]
})

kpis_gerais

,KPI,Valor
0,Total de Vendas,"1,000.0000"
1,Receita Total,"15,196,118.4787"
2,Lucro Total,"3,081,804.6787"
3,Margem de Lucro,0.2028
4,Quantidade Vendida,"6,062.0000"
5,Ticket Médio por Venda,"15,196.1185"
6,Desconto Médio,0.1282
7,Meta Total,"18,353,644.5200"
8,Taxa de Atingimento de Meta,0.0700
9,Taxa de Entregas Atrasadas,0.5190


In [4]:
# SEPARAÇÃO DOS ATRIBUTOS POR TIPO
# ============================================================

atributos_id_temporais = [
    "id_venda",
    "data",
    "mes",
    "trimestre"
]

atributos_qualitativos = [
    "cliente",
    "tipo_cliente",
    "faixa_renda",
    "segmento",
    "produto",
    "categoria",
    "marca",
    "linha_produto",
    "regiao",
    "cidade",
    "canal_venda",
    "vendedor",
    "atingiu_meta",
    "status_entrega"
]

atributos_quantitativos = [
    "tempo_relacionamento_meses",
    "quantidade",
    "preco_unitario",
    "desconto",
    "receita",
    "custo_unitario",
    "lucro",
    "meta_venda",
    "prazo_entrega_dias"
]

print("Atributos de identificação/temporais:")
print(atributos_id_temporais)

print("\nAtributos qualitativos:")
print(atributos_qualitativos)

print("\nAtributos quantitativos:")
print(atributos_quantitativos)

Atributos de identificação/temporais:
['id_venda', 'data', 'mes', 'trimestre']

Atributos qualitativos:
['cliente', 'tipo_cliente', 'faixa_renda', 'segmento', 'produto', 'categoria', 'marca', 'linha_produto', 'regiao', 'cidade', 'canal_venda', 'vendedor', 'atingiu_meta', 'status_entrega']

Atributos quantitativos:
['tempo_relacionamento_meses', 'quantidade', 'preco_unitario', 'desconto', 'receita', 'custo_unitario', 'lucro', 'meta_venda', 'prazo_entrega_dias']


# Análise descritiva dos dados qualitativos (nominais / ordinais)
segue abaixo

In [5]:
# RESUMO DOS ATRIBUTOS QUALITATIVOS
# ============================================================

resumo_qualitativo = []

for coluna in atributos_qualitativos:
    frequencias = df[coluna].value_counts(dropna=False)

    moda = frequencias.index[0]
    freq_abs_moda = frequencias.iloc[0]
    freq_rel_moda = freq_abs_moda / len(df)

    resumo_qualitativo.append({
        "atributo": coluna,
        "valores_unicos": df[coluna].nunique(dropna=True),
        "moda": moda,
        "frequencia_absoluta_moda": freq_abs_moda,
        "frequencia_relativa_moda": freq_rel_moda,
        "frequencia_relativa_moda_%": round(freq_rel_moda * 100, 2)
    })

resumo_qualitativo = pd.DataFrame(resumo_qualitativo)

resumo_qualitativo

,atributo,valores_unicos,moda,frequencia_absoluta_moda,frequencia_relativa_moda,frequencia_relativa_moda_%
0,cliente,199,Cliente_141,12,0.0120,1.2000
1,tipo_cliente,2,Recorrente,519,0.5190,51.9000
2,faixa_renda,3,Baixa,389,0.3890,38.9000
3,segmento,2,Atacado,508,0.5080,50.8000
4,produto,8,Monitor,143,0.1430,14.3000
5,categoria,4,Informática,492,0.4920,49.2000
6,marca,4,Alpha,255,0.2550,25.5000
7,linha_produto,3,Econômico,347,0.3470,34.7000
8,regiao,5,Norte,209,0.2090,20.9000
9,cidade,10,Manaus,116,0.1160,11.6000


In [6]:
# FUNÇÃO PARA TABELA DE FREQUÊNCIA QUALITATIVA
# ============================================================

def tabela_frequencia_qualitativa(base, coluna):
    """
    Gera a tabela de frequência de uma variável qualitativa.
    """
    # O parâmetro 'normalize=True' já calcula a frequência relativa
    tabela = base[coluna].value_counts(dropna=False, normalize=True).reset_index()
    tabela.columns = [coluna, "frequencia_relativa"]

    # Calcula a frequência absoluta a partir da relativa
    tabela["frequencia_absoluta"] = (tabela["frequencia_relativa"] * len(base)).astype(int)

    # Converte para porcentagem
    tabela["frequencia_relativa_%"] = (tabela["frequencia_relativa"] * 100).round(2)

    # Calcula a frequência acumulada
    tabela["frequencia_acumulada_%"] = (tabela["frequencia_relativa"].cumsum() * 100).round(2)

    # Reordena as colunas para um layout mais lógico
    return tabela[[coluna, "frequencia_absoluta", "frequencia_relativa_%", "frequencia_acumulada_%"]]   

# ============================================================
# TABELAS DE FREQUÊNCIA PARA TODOS OS ATRIBUTOS QUALITATIVOS
# ============================================================

for coluna in atributos_qualitativos:
    print(f"\n{'='*80}")
    print(f"Tabela de frequência do atributo: {coluna}")
    print(f"{'='*80}")

    tabela_freq = tabela_frequencia_qualitativa(df, coluna)

    # Remove a condição e exibe a tabela completa com um contexto temporário
    with pd.option_context('display.max_rows', None):
        display(tabela_freq)
    


Tabela de frequência do atributo: cliente


,cliente,frequencia_absoluta,frequencia_relativa_%,frequencia_acumulada_%
0,Cliente_141,12,1.2000,1.2000
1,Cliente_027,10,1.0000,2.2000
2,Cliente_109,9,0.9000,3.1000
3,Cliente_024,9,0.9000,4.0000
4,Cliente_016,9,0.9000,4.9000
5,Cliente_036,9,0.9000,5.8000
6,Cliente_115,9,0.9000,6.7000
7,Cliente_048,9,0.9000,7.6000
8,Cliente_188,9,0.9000,8.5000
9,Cliente_121,9,0.9000,9.4000



Tabela de frequência do atributo: tipo_cliente


,tipo_cliente,frequencia_absoluta,frequencia_relativa_%,frequencia_acumulada_%
0,Recorrente,519,51.9000,51.9000
1,Novo,481,48.1000,100.0000



Tabela de frequência do atributo: faixa_renda


...

,vendedor,frequencia_absoluta,frequencia_relativa_%,frequencia_acumulada_%
0,Vendedor_10,63,6.3000,6.3000
1,Vendedor_13,60,6.0000,12.3000
2,Vendedor_12,60,6.0000,18.3000
3,Vendedor_04,55,5.5000,23.8000
4,Vendedor_18,53,5.3000,29.1000
5,Vendedor_19,50,5.0000,34.1000
6,Vendedor_03,50,5.0000,39.1000
7,Vendedor_06,50,5.0000,44.1000
8,Vendedor_17,50,5.0000,49.1000
9,Vendedor_20,49,4.9000,54.0000



Tabela de frequência do atributo: atingiu_meta


,atingiu_meta,frequencia_absoluta,frequencia_relativa_%,frequencia_acumulada_%
0,Não,930,93.0000,93.0000
1,Sim,70,7.0000,100.0000



Tabela de frequência do atributo: status_entrega


,status_entrega,frequencia_absoluta,frequencia_relativa_%,frequencia_acumulada_%
0,Atrasado,519,51.9000,51.9000
1,No prazo,481,48.1000,100.0000


### Análise dos Atributos Qualitativos

Os atributos qualitativos foram analisados por meio da contagem de valores únicos, frequência absoluta, frequência relativa e moda. Esse tipo de análise foram escolhidas pois permite compreender a distribuição das categorias dentro da base.

A contagem de valores únicos permitiu identificar variáveis com baixa cardinalidade, como `tipo_cliente`, `segmento`, `canal_venda`, `atingiu_meta` e `status_entrega`, e variáveis com maior cardinalidade, como `cliente` e `vendedor`.

A frequência absoluta mostrou quantas vezes cada categoria aparece na base.  
A frequência relativa indicou a participação percentual de cada categoria em relação ao total de registros. 

A moda foi utilizada para identificar a categoria mais recorrente em cada atributo.

# Análise de Dados Quantitativos
segue abaixo

In [8]:
# RESUMO DOS ATRIBUTOS QUANTITATIVOS
# ============================================================

resumo_quantitativo = []

for coluna in atributos_quantitativos:
    serie = df[coluna].dropna()

    moda = serie.mode()
    moda_valor = moda.iloc[0] if not moda.empty else np.nan

    resumo_quantitativo.append({
        "atributo": coluna,
        "contagem": serie.count(),
        "media": serie.mean(),
        "mediana": serie.median(),
        "moda": moda_valor,
        "minimo": serie.min(),
        "maximo": serie.max(),
        "amplitude": serie.max() - serie.min(),
        "desvio_padrao": serie.std(),
        "coeficiente_variacao": serie.std() / serie.mean() if serie.mean() != 0 else np.nan,
        "q1_25%": serie.quantile(0.25),
        "q2_50%": serie.quantile(0.50),
        "q3_75%": serie.quantile(0.75),
        "iqr": serie.quantile(0.75) - serie.quantile(0.25),
        "p5": serie.quantile(0.05),
        "p10": serie.quantile(0.10),
        "p90": serie.quantile(0.90),
        "p95": serie.quantile(0.95),
        "assimetria": serie.skew(),
        "curtose": serie.kurt()
    })


resumo_quantitativo = pd.DataFrame(resumo_quantitativo)

resumo_quantitativo   

,atributo,contagem,media,mediana,moda,minimo,maximo,amplitude,desvio_padrao,coeficiente_variacao,q1_25%,q2_50%,q3_75%,iqr,p5,p10,p90,p95,assimetria,curtose
0,tempo_relacionamento_meses,1000,16.6660,5.0000,4.0000,0.0000,59.0000,59.0000,18.4595,1.1076,2.0000,5.0000,31.0000,29.0000,0.0000,1.0000,48.0000,53.0000,0.9009,-0.6850
1,quantidade,1000,6.0620,6.0000,7.0000,1.0000,11.0000,10.0000,3.1247,0.5155,3.0000,6.0000,9.0000,6.0000,1.0000,1.9000,10.0000,11.0000,-0.1075,-1.1669
2,preco_unitario,1000,"2,924.7153","2,780.8900","2,635.2600",81.4500,"6,738.4862","6,657.0362","1,726.8995",0.5905,"1,426.8100","2,780.8900","4,452.7450","3,025.9350",370.7425,645.1410,"5,393.0100","5,695.5800",0.1295,-1.2072
3,desconto,1000,0.1282,0.1300,0.1400,0.0000,0.2500,0.2500,0.0709,0.5534,0.0700,0.1300,0.1900,0.1200,0.0100,0.0300,0.2200,0.2400,-0.0999,-1.1326
4,receita,1000,"15,196.1185","11,199.2130",61.9020,61.9020,"64,052.0100","63,990.1080","12,773.9807",0.8406,"4,691.6275","11,199.2130","22,755.0274","18,063.3999","1,194.3333","2,092.4175","33,642.1061","42,041.8524",1.0452,0.4574
5,custo_unitario,1000,"2,034.8856","1,906.1200","1,308.2800",46.0600,"5,178.2000","5,132.1400","1,260.5210",0.6195,993.7825,"1,906.1200","3,038.5350","2,044.7525",255.9715,437.8890,"3,828.4890","4,150.0580",0.3284,-0.8906
6,lucro,1000,"3,081.8047","1,723.8235","-5,323.1080","-5,323.1080","22,781.8305","28,104.9385","4,065.4309",1.3192,437.4554,"1,723.8235","4,272.8677","3,835.4123",-556.5281,-54.5443,"8,432.4803","12,143.4831",1.8151,3.8263
7,meta_venda,1000,"18,353.6445","13,514.7750",87.4400,87.4400,"72,103.3700","72,015.9300","15,371.4257",0.8375,"5,614.7850","13,514.7750","27,480.4675","21,865.6825","1,423.3435","2,500.8380","41,139.8210","50,262.7150",1.0023,0.2899
8,prazo_entrega_dias,1000,7.6580,8.0000,14.0000,1.0000,14.0000,13.0000,4.1685,0.5443,4.0000,8.0000,11.0000,7.0000,1.0000,2.0000,13.0000,14.0000,-0.0519,-1.2613


In [9]:
# CLASSIFICAÇÃO DE ASSIMETRIA E CURTOSE
# ============================================================

def classificar_assimetria(valor):
    if valor > 1:
        return "Assimetria positiva forte"
    elif valor > 0.5:
        return "Assimetria positiva moderada"
    elif valor < -1:
        return "Assimetria negativa forte"
    elif valor < -0.5:
        return "Assimetria negativa moderada"
    else:
        return "Aproximadamente simétrica"


def classificar_curtose(valor):
    if valor > 1:
        return "Curtose elevada / caudas pesadas"
    elif valor < -1:
        return "Curtose baixa / distribuição achatada"
    else:
        return "Curtose próxima do padrão"


resumo_quantitativo["interpretacao_assimetria"] = resumo_quantitativo["assimetria"].apply(classificar_assimetria)
resumo_quantitativo["interpretacao_curtose"] = resumo_quantitativo["curtose"].apply(classificar_curtose)

resumo_quantitativo[
    [
        "atributo",
        "assimetria",
        "interpretacao_assimetria",
        "curtose",
        "interpretacao_curtose"
    ]
]

,atributo,assimetria,interpretacao_assimetria,curtose,interpretacao_curtose
0,tempo_relacionamento_meses,0.9009,Assimetria positiva moderada,-0.6850,Curtose próxima do padrão
1,quantidade,-0.1075,Aproximadamente simétrica,-1.1669,Curtose baixa / distribuição achatada
2,preco_unitario,0.1295,Aproximadamente simétrica,-1.2072,Curtose baixa / distribuição achatada
3,desconto,-0.0999,Aproximadamente simétrica,-1.1326,Curtose baixa / distribuição achatada
4,receita,1.0452,Assimetria positiva forte,0.4574,Curtose próxima do padrão
5,custo_unitario,0.3284,Aproximadamente simétrica,-0.8906,Curtose próxima do padrão
6,lucro,1.8151,Assimetria positiva forte,3.8263,Curtose elevada / caudas pesadas
7,meta_venda,1.0023,Assimetria positiva forte,0.2899,Curtose próxima do padrão
8,prazo_entrega_dias,-0.0519,Aproximadamente simétrica,-1.2613,Curtose baixa / distribuição achatada


In [10]:
# PERCENTIS DETALHADOS
# ============================================================

percentis = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

tabela_percentis = df[atributos_quantitativos].quantile(percentis).T

tabela_percentis.columns = [
    "p1",
    "p5",
    "p10",
    "p25",
    "p50",
    "p75",
    "p90",
    "p95",
    "p99"
]

tabela_percentis

,p1,p5,p10,p25,p50,p75,p90,p95,p99
tempo_relacionamento_meses,0.0000,0.0000,1.0000,2.0000,5.0000,31.0000,48.0000,53.0000,58.0000
quantidade,1.0000,1.0000,1.9000,3.0000,6.0000,9.0000,10.0000,11.0000,11.0000
preco_unitario,105.9441,370.7425,645.1410,"1,426.8100","2,780.8900","4,452.7450","5,393.0100","5,695.5800","5,932.5374"
desconto,0.0000,0.0100,0.0300,0.0700,0.1300,0.1900,0.2200,0.2400,0.2500
receita,361.8968,"1,194.3333","2,092.4175","4,691.6275","11,199.2130","22,755.0274","33,642.1061","42,041.8524","50,751.8328"
custo_unitario,72.6313,255.9715,437.8890,993.7825,"1,906.1200","3,038.5350","3,828.4890","4,150.0580","4,850.9862"
lucro,"-2,631.7259",-556.5281,-54.5443,437.4554,"1,723.8235","4,272.8677","8,432.4803","12,143.4831","17,968.3113"
meta_venda,367.7592,"1,423.3435","2,500.8380","5,614.7850","13,514.7750","27,480.4675","41,139.8210","50,262.7150","61,696.8640"
prazo_entrega_dias,1.0000,1.0000,2.0000,4.0000,8.0000,11.0000,13.0000,14.0000,14.0000


### Análise dos Atributos Quantitativos

Os atributos quantitativos foram analisados por meio de medidas de tendência central, posição, dispersão e formato da distribuição. Foram calculadas média, mediana, moda, mínimo, máximo, amplitude, desvio padrão, coeficiente de variação, quartis, percentis, assimetria e curtose.

A média foi utilizada para representar o valor médio das variáveis  
A mediana foi analisada como medida de posição central menos sensível a valores extremos.  
A moda foi calculada para identificar o valor mais frequente em cada atributo, sendo especialmente útil em variáveis discretas, como quantidade vendida e prazo de entrega.

Os quartis e percentis permitiram compreender a distribuição dos dados em diferentes pontos, identificando faixas inferiores, centrais e superiores.  
O intervalo interquartil foi utilizado para observar a dispersão da parte central da distribuição.  

A assimetria foi utilizada para avaliar se as distribuições são aproximadamente simétricas ou se possuem concentração em um dos lados. Já a curtose foi analisada para verificar a presença de caudas mais pesadas ou distribuições mais achatadas.

Além disso, foram gerados histogramas e boxplots para visualizar a distribuição dos atributos quantitativos. A matriz de correlação foi utilizada para identificar relações lineares entre variáveis numéricas, especialmente entre receita, lucro, desconto, quantidade, preço unitário e meta de venda.

# Atributos de Identificação e Temporais
id_venda

data, mes, trimestre

In [13]:
# VALIDAÇÃO DE IDENTIFICADOR DA VENDA
# ============================================================

total_linhas = df.shape[0]
ids_unicos = df["id_venda"].nunique()
ids_duplicados = df["id_venda"].duplicated().sum()
id_minimo = df["id_venda"].min()
id_maximo = df["id_venda"].max()

validacao_id = pd.DataFrame({
    "metrica": [
        "Total de registros",
        "IDs únicos",
        "IDs duplicados",
        "Menor ID",
        "Maior ID"
    ],
    "valor": [
        total_linhas,
        ids_unicos,
        ids_duplicados,
        id_minimo,
        id_maximo
    ]
})

validacao_id

,metrica,valor
0,Total de registros,1000
1,IDs únicos,1000
2,IDs duplicados,0
3,Menor ID,1
4,Maior ID,1000


In [15]:
# VALIDAÇÃO TEMPORAL
# ============================================================

df["data"] = pd.to_datetime(df["data"])

datas_invalidas = df["data"].isna().sum()
data_minima = df["data"].min()
data_maxima = df["data"].max()
quantidade_dias_unicos = df["data"].nunique()
periodo_total_dias = (data_maxima - data_minima).days + 1

validacao_temporal = pd.DataFrame({
    "metrica": [
        "Datas inválidas",
        "Data mínima",
        "Data máxima",
        "Quantidade de dias únicos com vendas",
        "Período total em dias"
    ],
    "valor": [
        datas_invalidas,
        data_minima,
        data_maxima,
        quantidade_dias_unicos,
        periodo_total_dias
    ]
})

validacao_temporal

,metrica,valor
0,Datas inválidas,0
1,Data mínima,2024-01-01 00:00:00
2,Data máxima,2024-12-30 00:00:00
3,Quantidade de dias únicos com vendas,338
4,Período total em dias,365


In [16]:
# RESUMO CONSOLIDADO DA ANÁLISE DESCRITIVA
# ============================================================

resumo_blocos = pd.DataFrame({
    "bloco": [
        "Atributos qualitativos",
        "Atributos quantitativos",
        "Atributos de identificação/temporais"
    ],
    "quantidade_atributos": [
        len(atributos_qualitativos),
        len(atributos_quantitativos),
        len(atributos_id_temporais)
    ],
    "analises_realizadas": [
        "Valores únicos, frequência absoluta, frequência relativa e moda",
        "Média, mediana, moda, quartis, percentis, assimetria, curtose, dispersão e correlação",
        "Unicidade, duplicidades, limites temporais, volume mensal/trimestral e consistência temporal"
    ]
})

resumo_blocos

,bloco,quantidade_atributos,analises_realizadas
0,Atributos qualitativos,14,"Valores únicos, frequência absoluta, frequênci..."
1,Atributos quantitativos,9,"Média, mediana, moda, quartis, percentis, assi..."
2,Atributos de identificação/temporais,4,"Unicidade, duplicidades, limites temporais, vo..."


## Etapa 4.1 - Análise Estatística Descritiva

A primeira parte da Etapa 4 consistiu na realização de uma análise estatística descritiva da base tratada. Para facilitar a compreensão, os atributos foram separados em três grupos: qualitativos, quantitativos e atributos de identificação/temporais.

Os atributos qualitativos foram analisados por meio da contagem de valores únicos, frequência absoluta, frequência relativa e moda. Essa abordagem permitiu compreender a distribuição das categorias presentes na base, identificando quais produtos, regiões, canais, vendedores, tipos de clientes e status de entrega aparecem com maior frequência. A moda foi utilizada para identificar a categoria predominante em cada variável.

Os atributos quantitativos foram analisados com medidas de tendência central, posição, dispersão e formato da distribuição. Foram calculadas média, mediana, moda, mínimo, máximo, amplitude, desvio padrão, coeficiente de variação, quartis, percentis, assimetria e curtose. Essas estatísticas permitiram compreender o comportamento numérico das variáveis de vendas, como receita, lucro, desconto, preço unitário, custo unitário, quantidade e prazo de entrega.

A análise dos quartis e percentis permitiu observar a distribuição dos valores em diferentes pontos, enquanto a assimetria e a curtose auxiliaram na identificação do formato das distribuições. Histogramas e boxplots foram utilizados para visualizar a distribuição dos atributos quantitativos, e a matriz de correlação foi utilizada para identificar relações lineares entre variáveis numéricas.

Os atributos de identificação e temporais foram analisados separadamente. O identificador `id_venda` foi validado quanto à unicidade, duplicidade e sequência. Já os atributos temporais foram avaliados por meio dos limites de data e período em dias.

Essa análise estatística descritiva forneceu uma visão geral da estrutura e do comportamento da base, servindo como base para as próximas análises da Etapa 4, que envolverão cruzamentos entre variáveis, identificação de padrões e interpretação de relações relevantes para o negócio.

# Cruzamento de Dados
- vendas pro mês
- vendas por trimestre
- vendas por dia
- 

In [17]:
# VOLUME DE VENDAS POR MÊS
# ============================================================

volume_mensal = df.groupby("mes").agg(
    quantidade_vendas=("id_venda", "count"),
    receita_total=("receita", "sum"),
    lucro_total=("lucro", "sum")
).reset_index()

volume_mensal["margem_lucro"] = (
    volume_mensal["lucro_total"] / volume_mensal["receita_total"]
)

volume_mensal

,mes,quantidade_vendas,receita_total,lucro_total,margem_lucro
0,1,93,"1,393,767.9698","258,583.4298",0.1855
1,2,84,"1,270,626.7478","276,598.5078",0.2177
2,3,98,"1,432,879.4623","236,255.5323",0.1649
3,4,81,"1,222,590.4084","234,821.8384",0.1921
4,5,81,"1,274,222.9590","314,610.7490",0.2469
5,6,70,"1,074,728.2723","204,117.6923",0.1899
6,7,80,"1,307,184.1022","283,004.5722",0.2165
7,8,70,"990,909.6458","209,072.2258",0.2110
8,9,69,"1,173,685.2836","219,670.5036",0.1872
9,10,86,"1,228,377.9629","236,113.7629",0.1922


In [18]:
# VOLUME DE VENDAS POR TRIMESTRE
# ============================================================

volume_trimestral = df.groupby("trimestre").agg(
    quantidade_vendas=("id_venda", "count"),
    receita_total=("receita", "sum"),
    lucro_total=("lucro", "sum")
).reset_index()

volume_trimestral["margem_lucro"] = (
    volume_trimestral["lucro_total"] / volume_trimestral["receita_total"]
)

volume_trimestral

,trimestre,quantidade_vendas,receita_total,lucro_total,margem_lucro
0,1,275,"4,097,274.1799","771,437.4699",0.1883
1,2,232,"3,571,541.6397","753,550.2797",0.2110
2,3,219,"3,471,779.0316","711,747.3016",0.2050
3,4,274,"4,055,523.6275","845,069.6275",0.2084


In [19]:
# VENDAS POR DIA DA SEMANA
# ============================================================

df["dia_semana"] = df["data"].dt.day_name()

ordem_dias = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

volume_dia_semana = df.groupby("dia_semana").agg(
    quantidade_vendas=("id_venda", "count"),
    receita_total=("receita", "sum"),
    lucro_total=("lucro", "sum")
).reindex(ordem_dias).reset_index()

volume_dia_semana["margem_lucro"] = (
    volume_dia_semana["lucro_total"] / volume_dia_semana["receita_total"]
)

volume_dia_semana

,dia_semana,quantidade_vendas,receita_total,lucro_total,margem_lucro
0,Monday,140,"2,119,048.5742","434,232.6642",0.2049
1,Tuesday,141,"2,499,405.4828","558,068.5628",0.2233
2,Wednesday,139,"1,978,996.2356","406,448.2956",0.2054
3,Thursday,136,"2,059,476.6460","412,863.6460",0.2005
4,Friday,159,"2,282,684.2910","452,758.7710",0.1983
5,Saturday,139,"2,146,386.4790","412,686.8090",0.1923
6,Sunday,146,"2,110,120.7701","404,745.9301",0.1918


In [20]:
# SÉRIE TEMPORAL DIÁRIA
# ============================================================

serie_diaria = df.groupby("data").agg(
    quantidade_vendas=("id_venda", "count"),
    receita_total=("receita", "sum"),
    lucro_total=("lucro", "sum")
).reset_index()

serie_diaria["margem_lucro"] = (
    serie_diaria["lucro_total"] / serie_diaria["receita_total"]
)

with pd.option_context('display.max_rows', None):
    display(serie_diaria)

,data,quantidade_vendas,receita_total,lucro_total,margem_lucro
0,2024-01-01,6,"137,202.0932","18,459.5032",0.1345
1,2024-01-02,5,"63,268.5190","10,673.4690",0.1687
2,2024-01-03,3,"29,941.2030","5,887.4330",0.1966
3,2024-01-04,1,"1,938.0900",109.1400,0.0563
4,2024-01-05,4,"87,904.1187","21,563.8987",0.2453
5,2024-01-06,3,"41,924.9950","2,223.8850",0.0530
6,2024-01-07,1,"30,696.5554","1,712.4954",0.0558
7,2024-01-08,1,"26,208.1818",335.7618,0.0128
8,2024-01-09,2,"4,728.9876",900.2676,0.1904
9,2024-01-10,3,"67,258.1816","4,556.3816",0.0677


# Análise Atributos e KPIs

[quantidade_vendas, quantidade_itens, receita, lucro, desconto_medio, media_total, taxa_meta, taxa_atraso, margem_lucro, ticket_medio]  = KPIs

- Região
- Canal de venda
- Categoria
- Produto
- Tipo de cliente
- Vendedor

In [22]:
# FUNÇÃO AUXILIAR PARA CRUZAMENTOS
# ============================================================

def analisar_dimensao(base, dimensao):
    """
    Agrupa a base por uma dimensão categórica e calcula os principais KPIs.
    """
    
    analise = base.groupby(dimensao, observed=True).agg(
        vendas=("id_venda", "count"),
        quantidade=("quantidade", "sum"),
        receita=("receita", "sum"),
        lucro=("lucro", "sum"),
        desconto_medio=("desconto", "mean"),
        meta_total=("meta_venda", "sum"),
        taxa_meta=("atingiu_meta", lambda x: (x == "Sim").mean()),
        taxa_atraso=("status_entrega", lambda x: (x == "Atrasado").mean())
    )
    
    analise["margem_lucro"] = analise["lucro"] / analise["receita"]
    analise["ticket_medio"] = analise["receita"] / analise["vendas"]
    
    return analise.sort_values("receita", ascending=False)

In [23]:
# 4.3 - ANÁLISE POR REGIÃO
# ============================================================

analise_regiao = analisar_dimensao(df, "regiao")

analise_regiao

,vendas,quantidade,receita,lucro,desconto_medio,meta_total,taxa_meta,taxa_atraso,margem_lucro,ticket_medio
regiao,,,,,,,,,,
Sul,209,1253,"3,314,346.5699","732,603.3099",0.1247,"3,984,848.2000",0.0478,0.5502,0.2210,"15,858.1176"
Centro-Oeste,202,1273,"3,265,071.8116","586,749.5016",0.1376,"4,006,791.2100",0.0545,0.4901,0.1797,"16,163.7218"
Norte,209,1278,"3,247,788.9366","689,213.0166",0.1186,"3,834,767.9300",0.1053,0.5215,0.2122,"15,539.6600"
Sudeste,199,1197,"2,720,463.9033","527,492.3733",0.1304,"3,322,269.1500",0.0653,0.5276,0.1939,"13,670.6729"
Nordeste,181,1061,"2,648,447.2573","545,746.4773",0.1304,"3,204,968.0300",0.0773,0.5028,0.2061,"14,632.3053"


In [24]:
# 4.4 - ANÁLISE POR CANAL DE VENDA
# ============================================================

analise_canal = analisar_dimensao(df, "canal_venda")

analise_canal

,vendas,quantidade,receita,lucro,desconto_medio,meta_total,taxa_meta,taxa_atraso,margem_lucro,ticket_medio
canal_venda,,,,,,,,,,
Online,522,3169,"8,092,223.7471","1,600,532.6771",0.1300,"9,830,537.0200",0.0747,0.4885,0.1978,"15,502.3443"
Loja,478,2893,"7,103,894.7316","1,481,272.0016",0.1263,"8,523,107.5000",0.0649,0.5523,0.2085,"14,861.7045"


In [25]:
# 4.5 - ANÁLISE POR CATEGORIA
# ============================================================

analise_categoria = analisar_dimensao(df, "categoria")

analise_categoria

,vendas,quantidade,receita,lucro,desconto_medio,meta_total,taxa_meta,taxa_atraso,margem_lucro,ticket_medio
categoria,,,,,,,,,,
Informática,492,3021,"7,297,489.8756","1,502,208.4156",0.1276,"8,764,772.1100",0.0813,0.5041,0.2059,"14,832.2965"
Periféricos,346,2029,"5,432,341.1166","1,091,976.2666",0.1269,"6,617,255.8400",0.0636,0.5491,0.2010,"15,700.4079"
Telefonia,132,800,"1,934,879.5352","399,791.5652",0.1345,"2,333,209.6900",0.0379,0.4924,0.2066,"14,658.1783"
Outros,30,212,"531,407.9513","87,828.4313",0.1250,"638,406.8800",0.1000,0.5333,0.1653,"17,713.5984"


In [26]:
# 4.6 - ANÁLISE POR PRODUTO
# ============================================================

analise_produto = analisar_dimensao(df, "produto")

analise_produto

,vendas,quantidade,receita,lucro,desconto_medio,meta_total,taxa_meta,taxa_atraso,margem_lucro,ticket_medio
produto,,,,,,,,,,
Monitor,143,960,"2,463,025.3196","448,537.8596",0.1306,"2,980,361.9600",0.1049,0.5245,0.1821,"17,223.9533"
Teclado,115,683,"1,973,461.4424","425,008.6724",0.1312,"2,372,065.2300",0.0522,0.5739,0.2154,"17,160.5343"
Mouse,133,796,"1,962,335.2508","389,571.5908",0.1176,"2,381,767.6700",0.0827,0.5338,0.1985,"14,754.4004"
Smartphone,133,805,"1,946,475.0212","402,535.9512",0.1347,"2,347,534.6400",0.0376,0.4887,0.2068,"14,635.1505"
Impressora,115,724,"1,859,469.9215","419,170.2815",0.1322,"2,259,276.6200",0.0609,0.5130,0.2254,"16,169.3037"
Headset,111,644,"1,697,277.7110","310,612.3510",0.1352,"2,100,652.0800",0.0541,0.5405,0.1830,"15,290.7902"
Notebook,122,719,"1,647,800.9065","386,693.2165",0.1185,"1,947,316.0100",0.0820,0.5246,0.2347,"13,506.5648"
Tablet,128,731,"1,646,272.9057","299,674.7557",0.1266,"1,964,670.3100",0.0781,0.4609,0.1820,"12,861.5071"


In [27]:
# 4.7 - ANÁLISE POR TIPO DE CLIENTE
# ============================================================

analise_tipo_cliente = analisar_dimensao(df, "tipo_cliente")

analise_tipo_cliente

,vendas,quantidade,receita,lucro,desconto_medio,meta_total,taxa_meta,taxa_atraso,margem_lucro,ticket_medio
tipo_cliente,,,,,,,,,,
Recorrente,519,3186,"7,853,078.6324","1,578,620.5424",0.1287,"9,532,590.8900",0.0809,0.4855,0.2010,"15,131.1727"
Novo,481,2876,"7,343,039.8463","1,503,184.1363",0.1276,"8,821,053.6300",0.0582,0.5551,0.2047,"15,266.1951"


In [28]:
# 4.8 - ANÁLISE POR VENDEDOR
# ============================================================

analise_vendedor = analisar_dimensao(df, "vendedor")

# Top 10 vendedores por receita
top_vendedores_receita = analise_vendedor.sort_values("receita", ascending=False)

top_vendedores_receita

,vendas,quantidade,receita,lucro,desconto_medio,meta_total,taxa_meta,taxa_atraso,margem_lucro,ticket_medio
vendedor,,,,,,,,,,
Vendedor_13,60,384,"1,050,462.6194","206,493.8594",0.1299,"1,249,363.0400",0.0667,0.5000,0.1966,"17,507.7103"
Vendedor_18,53,329,"958,083.4186","203,100.3786",0.1248,"1,163,375.2200",0.0755,0.3585,0.2120,"18,077.0456"
Vendedor_12,60,382,"939,076.5662","153,868.6462",0.1296,"1,177,067.8900",0.0333,0.4833,0.1639,"15,651.2761"
Vendedor_19,50,326,"908,888.4283","189,655.6283",0.1462,"1,119,304.1500",0.0600,0.5000,0.2087,"18,177.7686"
Vendedor_20,49,313,"905,934.0195","239,532.8195",0.1269,"1,101,882.5600",0.0612,0.5510,0.2644,"18,488.4494"
Vendedor_10,63,367,"832,485.6311","187,059.0811",0.1324,"984,022.4000",0.0317,0.6190,0.2247,"13,214.0576"
Vendedor_08,46,345,"809,109.5116","161,494.0416",0.1388,"985,587.5500",0.0217,0.6304,0.1996,"17,589.3372"
Vendedor_17,50,317,"800,704.4098","157,393.4398",0.1194,"937,353.4100",0.1400,0.4600,0.1966,"16,014.0882"
Vendedor_03,50,277,"787,685.1876","130,323.1076",0.1208,"924,067.0900",0.2000,0.5600,0.1655,"15,753.7038"


# Cruzamento - Respostas as Perguntas de Negócio



In [29]:
# ETAPA 4.4 - CORRELAÇÕES, CRUZAMENTOS E RELAÇÕES DE NEGÓCIO
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from matplotlib.ticker import FuncFormatter
from IPython.display import display

# Configurações
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:,.4f}".format)

sns.set_theme(style="whitegrid")

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10


# CARREGAMENTO DA BASE TRATADA
# ============================================================

caminho_base = Path("dados_empresa_tratado.csv")

if not caminho_base.exists():
    raise FileNotFoundError(
        "O arquivo 'dados_empresa_tratado.csv' não foi encontrado. "
        "Verifique se ele está na mesma pasta deste notebook."
    )

df = pd.read_csv(caminho_base)

# Converter data
df["data"] = pd.to_datetime(df["data"], errors="coerce")

# Criar variável numérica para atingimento de meta
df["atingiu_meta_flag"] = np.where(df["atingiu_meta"] == "Sim", 1, 0)

# Criar margem de lucro por venda
df["margem_lucro_venda"] = df["lucro"] / df["receita"]

In [30]:
# FUNÇÕES AUXILIARES
# ============================================================

def eixo_moeda(x, pos):
    if abs(x) >= 1_000_000:
        return f"R$ {x/1_000_000:.1f} mi"
    elif abs(x) >= 1_000:
        return f"R$ {x/1_000:.0f} mil"
    else:
        return f"R$ {x:.0f}"


def eixo_percentual(x, pos):
    return f"{x:.0%}"


def formatar_moeda(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def formatar_percentual(valor):
    return f"{valor:.2%}".replace(".", ",")

## Regiões vs. Receita e Margem de Lucro

Responder à pergunta: **Quais regiões geram maior receita e quais apresentam menor margem de lucro?**
    
Será calculado:  
- soma da receita;
- soma do lucro;
- margem de lucro agregada. (margem_lucro = lucro_total / receita_total)

In [31]:
# REGIÕES VS. RECEITA E MARGEM DE LUCRO
# ============================================================

analise_regiao = df.groupby("regiao").agg(
    quantidade_vendas=("id_venda", "count"),
    receita_total=("receita", "sum"),
    lucro_total=("lucro", "sum")
).reset_index()

analise_regiao["margem_lucro"] = (
    analise_regiao["lucro_total"] / analise_regiao["receita_total"]
)

analise_regiao = analise_regiao.sort_values("receita_total", ascending=False)

analise_regiao

,regiao,quantidade_vendas,receita_total,lucro_total,margem_lucro
4,Sul,209,"3,314,346.5699","732,603.3099",0.2210
0,Centro-Oeste,202,"3,265,071.8116","586,749.5016",0.1797
2,Norte,209,"3,247,788.9366","689,213.0166",0.2122
3,Sudeste,199,"2,720,463.9033","527,492.3733",0.1939
1,Nordeste,181,"2,648,447.2573","545,746.4773",0.2061


## Produtos e Categorias vs. Volume e Lucro

Responder à pergunta: **Quais produtos e categorias são mais vendidos e quais geram maior lucro?**  

Será calculado:
- soma da quantidade;
- soma do lucro.

In [32]:
# PRODUTOS E CATEGORIAS VS. VOLUME E LUCRO
# ============================================================

import pandas as pd

analise_categoria = (
    df.groupby("categoria")
    .agg(quantidade_total=("quantidade", "sum"), lucro_total=("lucro", "sum"))
    .reset_index()
)

analise_categoria = analise_categoria.sort_values(
    "lucro_total", ascending=False
).round(2)

analise_produto = (
    df.groupby("produto")
    .agg(quantidade_total=("quantidade", "sum"), lucro_total=("lucro", "sum"))
    .reset_index()
)

analise_produto = analise_produto.sort_values(
    "lucro_total", ascending=False
).round(2)

print("--- DESEMPENHO POR CATEGORIA ---")
display(analise_categoria) 

print("\n--- DESEMPENHO POR PRODUTO ---")
display(analise_produto)

analise_categoria_produto = df.groupby(["categoria", "produto"]).agg(
    quantidade_total=("quantidade", "sum"),
    lucro_total=("lucro", "sum")
).reset_index()

analise_categoria_produto = analise_categoria_produto.sort_values(
    "lucro_total",
    ascending=False
)

print("\n--- DESEMPENHO POR CATEGORIA-PRODUTO ---")
analise_categoria_produto

--- DESEMPENHO POR CATEGORIA ---


,categoria,quantidade_total,lucro_total
0,Informática,3021,"1,502,208.4200"
2,Periféricos,2029,"1,091,976.2700"
3,Telefonia,800,"399,791.5700"
1,Outros,212,"87,828.4300"



--- DESEMPENHO POR PRODUTO ---


,produto,quantidade_total,lucro_total
2,Monitor,960,"448,537.8600"
7,Teclado,683,"425,008.6700"
1,Impressora,724,"419,170.2800"
5,Smartphone,805,"402,535.9500"
3,Mouse,796,"389,571.5900"
4,Notebook,719,"386,693.2200"
0,Headset,644,"310,612.3500"
6,Tablet,731,"299,674.7600"



--- DESEMPENHO POR CATEGORIA-PRODUTO ---


,categoria,produto,quantidade_total,lucro_total
1,Informática,Monitor,931,"440,159.9034"
14,Periféricos,Teclado,629,"413,047.0036"
0,Informática,Impressora,688,"410,719.7664"
15,Telefonia,Smartphone,800,"399,791.5652"
13,Periféricos,Mouse,769,"372,363.3626"
2,Informática,Notebook,685,"369,480.7685"
12,Periféricos,Headset,631,"306,565.9004"
3,Informática,Tablet,717,"281,847.9773"
10,Outros,Tablet,14,"17,826.7784"
8,Outros,Notebook,34,"17,212.4480"


## Canal de Venda — Online vs. Loja Física

Responde à pergunta: **O canal online apresenta desempenho melhor ou pior que a loja física?**  

Agrupar por canal_venda e analisar:  
- receita total;
- lucro total;
- média de atingiu_meta.  
Como atingiu_meta é textual, usamos a versão numérica atingiu_meta_flag: Sim = 1 | Não = 0  
A média dessa variável representa a taxa de atingimento de meta.

In [33]:
# CANAL DE VENDA VS. RECEITA, LUCRO E META
# ============================================================

analise_canal = df.groupby("canal_venda").agg(
    quantidade_vendas=("id_venda", "count"),
    receita_total=("receita", "sum"),
    lucro_total=("lucro", "sum"),
    taxa_atingiu_meta=("atingiu_meta_flag", "mean")
).reset_index()

analise_canal["margem_lucro"] = (
    analise_canal["lucro_total"] / analise_canal["receita_total"]
)

analise_canal = analise_canal.sort_values("receita_total", ascending=False)

analise_canal

,canal_venda,quantidade_vendas,receita_total,lucro_total,taxa_atingiu_meta,margem_lucro
1,Online,522,"8,092,223.7471","1,600,532.6771",0.0747,0.1978
0,Loja,478,"7,103,894.7316","1,481,272.0016",0.0649,0.2085


## Desempenho da Equipe Comercial — Vendedores

Responde à pergunta: **Quais vendedores possuem melhor desempenho em receita, lucro e cumprimento de metas?**  

Agrupa por vendedor, calcula:
- soma da receita;
- soma do lucro;
- porcentagem de atingimento de meta.

In [34]:
# DESEMPENHO DA EQUIPE COMERCIAL
# ============================================================

analise_vendedor = df.groupby("vendedor").agg(
    quantidade_vendas=("id_venda", "count"),
    receita_total=("receita", "sum"),
    lucro_total=("lucro", "sum"),
    taxa_atingiu_meta=("atingiu_meta_flag", "mean")
).reset_index()

analise_vendedor["margem_lucro"] = (
    analise_vendedor["lucro_total"] / analise_vendedor["receita_total"]
)

analise_vendedor = analise_vendedor.sort_values(
    "lucro_total",
    ascending=False
)

analise_vendedor

,vendedor,quantidade_vendas,receita_total,lucro_total,taxa_atingiu_meta,margem_lucro
19,Vendedor_20,49,"905,934.0195","239,532.8195",0.0612,0.2644
12,Vendedor_13,60,"1,050,462.6194","206,493.8594",0.0667,0.1966
17,Vendedor_18,53,"958,083.4186","203,100.3786",0.0755,0.2120
3,Vendedor_04,55,"783,594.9500","193,736.3400",0.1273,0.2472
18,Vendedor_19,50,"908,888.4283","189,655.6283",0.0600,0.2087
9,Vendedor_10,63,"832,485.6311","187,059.0811",0.0317,0.2247
5,Vendedor_06,50,"733,731.7434","165,302.8134",0.0200,0.2253
7,Vendedor_08,46,"809,109.5116","161,494.0416",0.0217,0.1996
1,Vendedor_02,48,"723,128.0519","159,101.6319",0.1667,0.2200
16,Vendedor_17,50,"800,704.4098","157,393.4398",0.1400,0.1966


## Fidelidade do Cliente — Recorrentes vs. Novos

Responde à pergunta: **Clientes recorrentes geram maior receita e lucro do que clientes novos?**  

Agrupa por tipo_cliente, calcula:
- soma da receita;
- soma do lucro;
- margem de lucro.

In [35]:
# FIDELIDADE DO CLIENTE - RECORRENTES VS. NOVOS
# ============================================================

analise_tipo_cliente = df.groupby("tipo_cliente").agg(
    quantidade_vendas=("id_venda", "count"),
    receita_total=("receita", "sum"),
    lucro_total=("lucro", "sum")
).reset_index()

analise_tipo_cliente["margem_lucro"] = (
    analise_tipo_cliente["lucro_total"] / analise_tipo_cliente["receita_total"]
)

analise_tipo_cliente = analise_tipo_cliente.sort_values(
    "receita_total",
    ascending=False
)

analise_tipo_cliente

,tipo_cliente,quantidade_vendas,receita_total,lucro_total,margem_lucro
1,Recorrente,519,"7,853,078.6324","1,578,620.5424",0.2010
0,Novo,481,"7,343,039.8463","1,503,184.1363",0.2047


## Impacto dos Descontos na Margem de Lucro

Responde à pergunta: **O nível de desconto concedido impacta negativamente a margem de lucro?**  

Correlação estatística real, calcular:
- correlação de Pearson, que mede relação linear.
- correlação de Spearman, que mede relação monotônica, mesmo que não seja perfeitamente linear.

In [36]:
# IMPACTO DOS DESCONTOS NA MARGEM DE LUCRO
# ============================================================

# 1. Garante a criação da coluna de margem de lucro
df["margem_lucro_venda"] = np.where(
    df["receita"] != 0, df["lucro"] / df["receita"], 0
)


# 2. Cria uma regra simples para classificar o tamanho do desconto dado na venda
def agrupar_desconto(valor):
    if valor == 0:
        return "1. Sem Desconto (0%)"
    elif valor <= 0.10:
        return "2. Desconto Baixo (Até 10%)"
    elif valor <= 0.20:
        return "3. Desconto Moderado (11% a 20%)"
    else:
        return "4. Desconto Alto (Acima de 20%)"


# Aplica a regra na base de dados
df["Faixa de Desconto"] = df["desconto"].apply(agrupar_desconto)

# 3. Agrupa os dados para ver o impacto real na margem e no lucro médio
relatorio_simples = (
    df.groupby("Faixa de Desconto")
    .agg(
        total_vendas=("id_venda", "count"),
        desconto_medio=("desconto", "mean"),
        margem_lucro_media=("margem_lucro_venda", "mean"),
        lucro_medio_por_venda=("lucro", "mean"),
    )
    .reset_index()
)

# Formata as colunas para porcentagem e moeda para ficar fácil de ler
relatorio_simples["desconto_medio"] = (
    relatorio_simples["desconto_medio"] * 100
).round(1).astype(str) + "%"
relatorio_simples["margem_lucro_media"] = (
    relatorio_simples["margem_lucro_media"] * 100
).round(1).astype(str) + "%"
relatorio_simples["lucro_medio_por_venda"] = (
    "R$ " + relatorio_simples["lucro_medio_por_venda"].round(2).astype(str)
)

relatorio_simples

,Faixa de Desconto,total_vendas,desconto_medio,margem_lucro_media,lucro_medio_por_venda
0,1. Sem Desconto (0%),20,0.0%,29.4%,R$ 6065.05
1,2. Desconto Baixo (Até 10%),365,5.5%,26.9%,R$ 4246.77
2,3. Desconto Moderado (11% a 20%),437,15.5%,17.0%,R$ 2736.42
3,4. Desconto Alto (Acima de 20%),178,22.7%,10.2%,R$ 1205.71


# Análise de Pareto 80/20

In [42]:
# ETAPA 4.3 - ANÁLISE DE PARETO 80/20
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from matplotlib.ticker import FuncFormatter
from IPython.display import display

# Configurações
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:,.4f}".format)

sns.set_theme(style="whitegrid")

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10


# CARREGAMENTO DA BASE TRATADA
# ============================================================

caminho_base = Path("dados_empresa_tratado.csv")

if not caminho_base.exists():
    raise FileNotFoundError(
        "O arquivo 'dados_empresa_tratado.csv' não foi encontrado. "
        "Verifique se ele está na mesma pasta deste notebook."
    )

df = pd.read_csv(caminho_base)

df["data"] = pd.to_datetime(df["data"], errors="coerce")

In [43]:
# FUNÇÕES AUXILIARES
# ============================================================

def eixo_moeda(x, pos):
    if abs(x) >= 1_000_000:
        return f"R$ {x/1_000_000:.1f} mi"
    elif abs(x) >= 1_000:
        return f"R$ {x/1_000:.0f} mil"
    else:
        return f"R$ {x:.0f}"


def eixo_percentual(x, pos):
    return f"{x:.0f}%"

In [44]:
# FUNÇÃO GERAL PARA ANÁLISE DE PARETO
# ============================================================

def analise_pareto(base, dimensao, metrica, nome_metrica=None):
    """
    Realiza análise de Pareto 80/20 para uma dimensão e uma métrica.

    Parâmetros:
    - base: DataFrame com os dados.
    - dimensao: coluna categórica que será agrupada.
    - metrica: coluna numérica que será somada.
    - nome_metrica: nome opcional para exibição.

    Retorna:
    - DataFrame com total, participação percentual, acumulado e grupo Pareto.
    """

    if nome_metrica is None:
        nome_metrica = metrica

    tabela = base.groupby(dimensao).agg(
        valor_total=(metrica, "sum"),
        quantidade_registros=("id_venda", "count")
    ).reset_index()

    # Ordenar do maior para o menor valor
    tabela = tabela.sort_values("valor_total", ascending=False).reset_index(drop=True)

    # Criar ranking
    tabela.insert(0, "ranking", range(1, len(tabela) + 1))

    # Calcular participação percentual
    total_metrica = tabela["valor_total"].sum()

    tabela["participacao_%"] = (
        tabela["valor_total"] / total_metrica * 100
    )

    # Calcular participação acumulada
    tabela["participacao_acumulada_%"] = tabela["participacao_%"].cumsum()

    # Identificar o ponto em que o acumulado atinge ou ultrapassa 80%
    indice_limite_80 = tabela[tabela["participacao_acumulada_%"] >= 80].index.min()

    tabela["grupo_pareto_80"] = np.where(
        tabela.index <= indice_limite_80,
        "Sim",
        "Não"
    )

    tabela["dimensao_analisada"] = dimensao
    tabela["metrica_analisada"] = nome_metrica

    # Reordenar colunas
    tabela = tabela[
        [
            "ranking",
            "dimensao_analisada",
            dimensao,
            "metrica_analisada",
            "valor_total",
            "quantidade_registros",
            "participacao_%",
            "participacao_acumulada_%",
            "grupo_pareto_80"
        ]
    ]

    return tabela

In [45]:
# PARETO - PRODUTO VS. LUCRO TOTAL
# ============================================================

pareto_produto_lucro = analise_pareto(
    base=df,
    dimensao="produto",
    metrica="lucro",
    nome_metrica="Lucro Total"
)

pareto_produto_lucro

,ranking,dimensao_analisada,produto,metrica_analisada,valor_total,quantidade_registros,participacao_%,participacao_acumulada_%,grupo_pareto_80
0,1,produto,Monitor,Lucro Total,"448,537.8596",143,14.5544,14.5544,Sim
1,2,produto,Teclado,Lucro Total,"425,008.6724",115,13.7909,28.3453,Sim
2,3,produto,Impressora,Lucro Total,"419,170.2815",115,13.6015,41.9467,Sim
3,4,produto,Smartphone,Lucro Total,"402,535.9512",133,13.0617,55.0084,Sim
4,5,produto,Mouse,Lucro Total,"389,571.5908",133,12.6410,67.6495,Sim
5,6,produto,Notebook,Lucro Total,"386,693.2165",122,12.5476,80.1971,Sim
6,7,produto,Headset,Lucro Total,"310,612.3510",111,10.0789,90.2760,Não
7,8,produto,Tablet,Lucro Total,"299,674.7557",128,9.7240,100.0000,Não


A análise mostra que o lucro está relativamente distribuído entre os produtos, porque são necessários 6 produtos de 8 para atingir cerca de 80% do lucro.  
Isso indica que a empresa não depende de apenas um produto para gerar lucro. Porém, os produtos no grupo Pareto devem receber atenção estratégica, pois concentram a maior parte do resultado financeiro.

In [46]:
# PARETO - CATEGORIA VS. LUCRO TOTAL
# ============================================================

pareto_categoria_lucro = analise_pareto(
    base=df,
    dimensao="categoria",
    metrica="lucro",
    nome_metrica="Lucro Total"
)

pareto_categoria_lucro

,ranking,dimensao_analisada,categoria,metrica_analisada,valor_total,quantidade_registros,participacao_%,participacao_acumulada_%,grupo_pareto_80
0,1,categoria,Informática,Lucro Total,"1,502,208.4156",492,48.7444,48.7444,Sim
1,2,categoria,Periféricos,Lucro Total,"1,091,976.2666",346,35.4330,84.1775,Sim
2,3,categoria,Telefonia,Lucro Total,"399,791.5652",132,12.9726,97.1501,Não
3,4,categoria,Outros,Lucro Total,"87,828.4313",30,2.8499,100.0000,Não


Apenas 2 de 4 categorias concentram mais de 80% do lucro total. Isso indica que a empresa depende fortemente dessas categorias para gerar resultado.  
A empresa deve priorizar ações comerciais, estoque e campanhas principalmente em Informática e Periféricos, sem ignorar oportunidades de melhoria em categorias menores.

In [47]:
# PARETO - REGIÃO VS. LUCRO TOTAL
# ============================================================

pareto_regiao_lucro = analise_pareto(
    base=df,
    dimensao="regiao",
    metrica="lucro",
    nome_metrica="Lucro Total"
)

pareto_regiao_lucro

,ranking,dimensao_analisada,regiao,metrica_analisada,valor_total,quantidade_registros,participacao_%,participacao_acumulada_%,grupo_pareto_80
0,1,regiao,Sul,Lucro Total,"732,603.3099",209,23.7719,23.7719,Sim
1,2,regiao,Norte,Lucro Total,"689,213.0166",209,22.3639,46.1358,Sim
2,3,regiao,Centro-Oeste,Lucro Total,"586,749.5016",202,19.0392,65.1750,Sim
3,4,regiao,Nordeste,Lucro Total,"545,746.4773",181,17.7087,82.8837,Sim
4,5,regiao,Sudeste,Lucro Total,"527,492.3733",199,17.1163,100.0000,Não


A concentração regional do lucro não é tão forte. O lucro está relativamente distribuído entre as regiões.  
Mesmo assim, as regiões Sul, Norte, Centro-Oeste e Nordeste formam o grupo principal de contribuição para o lucro.

In [48]:
# PARETO - VENDEDOR VS. LUCRO TOTAL
# ============================================================

pareto_vendedor_lucro = analise_pareto(
    base=df,
    dimensao="vendedor",
    metrica="lucro",
    nome_metrica="Lucro Total"
)

pareto_vendedor_lucro

,ranking,dimensao_analisada,vendedor,metrica_analisada,valor_total,quantidade_registros,participacao_%,participacao_acumulada_%,grupo_pareto_80
0,1,vendedor,Vendedor_20,Lucro Total,"239,532.8195",49,7.7725,7.7725,Sim
1,2,vendedor,Vendedor_13,Lucro Total,"206,493.8594",60,6.7004,14.4729,Sim
2,3,vendedor,Vendedor_18,Lucro Total,"203,100.3786",53,6.5903,21.0632,Sim
3,4,vendedor,Vendedor_04,Lucro Total,"193,736.3400",55,6.2865,27.3497,Sim
4,5,vendedor,Vendedor_19,Lucro Total,"189,655.6283",50,6.1540,33.5037,Sim
5,6,vendedor,Vendedor_10,Lucro Total,"187,059.0811",63,6.0698,39.5735,Sim
6,7,vendedor,Vendedor_06,Lucro Total,"165,302.8134",50,5.3638,44.9373,Sim
7,8,vendedor,Vendedor_08,Lucro Total,"161,494.0416",46,5.2402,50.1776,Sim
8,9,vendedor,Vendedor_02,Lucro Total,"159,101.6319",48,5.1626,55.3402,Sim
9,10,vendedor,Vendedor_17,Lucro Total,"157,393.4398",50,5.1072,60.4474,Sim


Neste caso, aproximadamente 15 dos 20 vendedores explicam cerca de 80% do lucro total.  
Isso mostra que o lucro da equipe comercial está relativamente distribuído, e não concentrado em poucos vendedores.

In [49]:
# PARETO - MARCA VS. LUCRO TOTAL
# ============================================================

pareto_marca_lucro = analise_pareto(
    base=df,
    dimensao="marca",
    metrica="lucro",
    nome_metrica="Lucro Total"
)

pareto_marca_lucro

,ranking,dimensao_analisada,marca,metrica_analisada,valor_total,quantidade_registros,participacao_%,participacao_acumulada_%,grupo_pareto_80
0,1,marca,Alpha,Lucro Total,"944,584.3441",255,30.6504,30.6504,Sim
1,2,marca,Beta,Lucro Total,"731,897.1498",250,23.7490,54.3993,Sim
2,3,marca,Gamma,Lucro Total,"726,318.2564",252,23.5680,77.9673,Sim
3,4,marca,Delta,Lucro Total,"679,004.9284",243,22.0327,100.0000,Sim


As marcas possuem contribuição relativamente equilibrada.  
A marca Alpha lidera a geração de lucro, mas não domina sozinha o resultado. Para atingir 80%, praticamente todas as marcas precisam ser consideradas.  
Isso indica que a empresa possui baixa dependência de uma única marca.

In [50]:
# PARETO - LINHA DE PRODUTO VS. LUCRO TOTAL
# ============================================================

pareto_linha_lucro = analise_pareto(
    base=df,
    dimensao="linha_produto",
    metrica="lucro",
    nome_metrica="Lucro Total"
)

pareto_linha_lucro

,ranking,dimensao_analisada,linha_produto,metrica_analisada,valor_total,quantidade_registros,participacao_%,participacao_acumulada_%,grupo_pareto_80
0,1,linha_produto,Econômico,Lucro Total,"1,052,581.7104",347,34.1547,34.1547,Sim
1,2,linha_produto,Premium,Lucro Total,"1,024,949.9764",330,33.2581,67.4128,Sim
2,3,linha_produto,Padrão,Lucro Total,"1,004,272.9919",323,32.5872,100.0000,Sim


As três linhas contribuem de forma bastante equilibrada para o lucro total.  
Não existe uma concentração forte em uma única linha. Isso mostra que a empresa consegue gerar lucro tanto em produtos econômicos quanto em produtos premium e padrão.

In [56]:
# PARETO - CLIENTE VS. LUCRO TOTAL
# ============================================================

pareto_cliente_lucro = analise_pareto(
    base=df,
    dimensao="cliente",
    metrica="lucro",
    nome_metrica="Lucro Total"
)

pareto_cliente_lucro

,ranking,dimensao_analisada,cliente,metrica_analisada,valor_total,quantidade_registros,participacao_%,participacao_acumulada_%,grupo_pareto_80
0,1,cliente,Cliente_027,Lucro Total,"51,569.9819",10,1.6734,1.6734,Sim
1,2,cliente,Cliente_092,Lucro Total,"49,858.0329",8,1.6178,3.2912,Sim
2,3,cliente,Cliente_163,Lucro Total,"45,361.2070",8,1.4719,4.7631,Sim
3,4,cliente,Cliente_095,Lucro Total,"42,719.3306",7,1.3862,6.1493,Sim
4,5,cliente,Cliente_144,Lucro Total,"42,483.6581",8,1.3785,7.5278,Sim
5,6,cliente,Cliente_017,Lucro Total,"41,843.4688",6,1.3578,8.8856,Sim
6,7,cliente,Cliente_024,Lucro Total,"40,736.2757",9,1.3218,10.2074,Sim
7,8,cliente,Cliente_041,Lucro Total,"39,215.1878",7,1.2725,11.4799,Sim
8,9,cliente,Cliente_179,Lucro Total,"38,882.2346",6,1.2617,12.7415,Sim
9,10,cliente,Cliente_033,Lucro Total,"38,873.6509",8,1.2614,14.0029,Sim


In [51]:
# PARETO - CLIENTE VS. RECEITA TOTAL
# ============================================================

pareto_cliente_receita = analise_pareto(
    base=df,
    dimensao="cliente",
    metrica="receita",
    nome_metrica="Receita Total"
)

pareto_cliente_receita

,ranking,dimensao_analisada,cliente,metrica_analisada,valor_total,quantidade_registros,participacao_%,participacao_acumulada_%,grupo_pareto_80
0,1,cliente,Cliente_141,Receita Total,"230,739.4268",12,1.5184,1.5184,Sim
1,2,cliente,Cliente_163,Receita Total,"203,101.4770",8,1.3365,2.8549,Sim
2,3,cliente,Cliente_027,Receita Total,"189,974.1819",10,1.2501,4.1051,Sim
3,4,cliente,Cliente_057,Receita Total,"187,579.4158",8,1.2344,5.3395,Sim
4,5,cliente,Cliente_041,Receita Total,"187,398.9478",7,1.2332,6.5727,Sim
5,6,cliente,Cliente_144,Receita Total,"179,184.7981",8,1.1791,7.7518,Sim
6,7,cliente,Cliente_092,Receita Total,"177,111.1229",8,1.1655,8.9173,Sim
7,8,cliente,Cliente_017,Receita Total,"171,863.6188",6,1.1310,10.0483,Sim
8,9,cliente,Cliente_118,Receita Total,"170,630.9178",6,1.1229,11.1712,Sim
9,10,cliente,Cliente_024,Receita Total,"167,628.3157",9,1.1031,12.2743,Sim


In [58]:
# PARETO - PRODUTO VS. RECEITA TOTAL
# ============================================================

pareto_produto_receita = analise_pareto(
    base=df,
    dimensao="produto",
    metrica="receita",
    nome_metrica="Receita Total"
)

pareto_produto_receita

,ranking,dimensao_analisada,produto,metrica_analisada,valor_total,quantidade_registros,participacao_%,participacao_acumulada_%,grupo_pareto_80
0,1,produto,Monitor,Receita Total,"2,463,025.3196",143,16.2083,16.2083,Sim
1,2,produto,Teclado,Receita Total,"1,973,461.4424",115,12.9866,29.1949,Sim
2,3,produto,Mouse,Receita Total,"1,962,335.2508",133,12.9134,42.1083,Sim
3,4,produto,Smartphone,Receita Total,"1,946,475.0212",133,12.8090,54.9173,Sim
4,5,produto,Impressora,Receita Total,"1,859,469.9215",115,12.2365,67.1538,Sim
5,6,produto,Headset,Receita Total,"1,697,277.7110",111,11.1692,78.3229,Sim
6,7,produto,Notebook,Receita Total,"1,647,800.9065",122,10.8436,89.1665,Sim
7,8,produto,Tablet,Receita Total,"1,646,272.9057",128,10.8335,100.0000,Não


In [57]:
# RESUMO CONSOLIDADO DAS ANÁLISES DE PARETO
# ============================================================

analises_pareto = {
    "Produto vs. Lucro": pareto_produto_lucro,
    "Categoria vs. Lucro": pareto_categoria_lucro,
    "Região vs. Lucro": pareto_regiao_lucro,
    "Vendedor vs. Lucro": pareto_vendedor_lucro,
    "Marca vs. Lucro": pareto_marca_lucro,
    "Linha de Produto vs. Lucro": pareto_linha_lucro,
    "Cliente vs. Receita": pareto_cliente_receita,
    "Cliente vs. Lucro": pareto_cliente_lucro,
    "Produto vs. Receita": pareto_produto_receita
}

resumo_pareto = []

for nome_analise, tabela in analises_pareto.items():
    grupo_80 = tabela[tabela["grupo_pareto_80"] == "Sim"]

    resumo_pareto.append({
        "analise": nome_analise,
        "total_itens": tabela.shape[0],
        "itens_no_grupo_80": grupo_80.shape[0],
        "percentual_itens_no_grupo_80": grupo_80.shape[0] / tabela.shape[0] * 100,
        "participacao_acumulada_grupo_80": grupo_80["participacao_acumulada_%"].max()
    })

resumo_pareto = pd.DataFrame(resumo_pareto)

resumo_pareto

,analise,total_itens,itens_no_grupo_80,percentual_itens_no_grupo_80,participacao_acumulada_grupo_80
0,Produto vs. Lucro,8,6,75.0000,80.1971
1,Categoria vs. Lucro,4,2,50.0000,84.1775
2,Região vs. Lucro,5,4,80.0000,82.8837
3,Vendedor vs. Lucro,20,15,75.0000,82.9963
4,Marca vs. Lucro,4,4,100.0000,100.0000
5,Linha de Produto vs. Lucro,3,3,100.0000,100.0000
6,Cliente vs. Receita,199,118,59.2965,80.3856
7,Cliente vs. Lucro,199,102,51.2563,80.0956
8,Produto vs. Receita,8,7,87.5000,89.1665


## Etapa 4.3 - Análise de Pareto 80/20

Nesta etapa foi aplicada a Análise de Pareto, também conhecida como regra 80/20, com o objetivo de identificar quais elementos concentram a maior parte dos resultados da empresa. A lógica da análise consiste em ordenar os itens do maior para o menor valor, calcular a participação percentual de cada item no total e, em seguida, calcular a participação acumulada.

Foram realizadas análises de Pareto para diferentes dimensões do negócio, como produto, categoria, região, vendedor, marca, linha de produto e cliente. As principais métricas utilizadas foram lucro total e receita total.

Na análise de produto versus lucro total, observou-se que seis dos oito produtos concentram aproximadamente 80% do lucro da empresa. Isso indica que o lucro está relativamente distribuído entre os produtos, embora alguns itens tenham maior peso no resultado financeiro.

Na análise por categoria, o comportamento de Pareto apareceu de forma mais clara. As categorias Informática e Periféricos concentraram aproximadamente 84% do lucro total. Isso mostra que essas duas categorias são as principais responsáveis pela geração de resultado financeiro da empresa e devem ser priorizadas em estratégias comerciais, campanhas e gestão de estoque.

Na análise regional, verificou-se que quatro das cinco regiões são necessárias para atingir cerca de 80% do lucro. Isso indica que o lucro está distribuído geograficamente, sem dependência extrema de uma única região.

Na análise por vendedor, aproximadamente quinze dos vinte vendedores explicam cerca de 80% do lucro. Esse resultado mostra que o desempenho comercial está relativamente distribuído entre a equipe, embora os vendedores mais bem posicionados devam ser analisados para identificação de boas práticas.

A análise por marca mostrou que a marca Alpha possui a maior participação no lucro, mas as demais marcas também possuem contribuição relevante. Portanto, não há dependência extrema de uma única marca. Já na análise por linha de produto, as linhas Econômico, Premium e Padrão apresentaram contribuições próximas, indicando equilíbrio entre os diferentes padrões de produto.

Também foi realizada a análise de Pareto por cliente em relação à receita e ao lucro. Essa análise é importante para avaliar concentração da carteira. Quando poucos clientes concentram grande parte da receita ou do lucro, existe maior risco comercial. Quando muitos clientes são necessários para atingir 80%, a empresa possui uma carteira mais distribuída.

Conclui-se que a Análise de Pareto permitiu identificar os principais focos estratégicos da empresa. As categorias Informática e Periféricos são os grupos mais relevantes para o lucro, enquanto produtos, regiões, marcas e vendedores apresentam distribuição mais equilibrada. Essa análise apoia decisões sobre priorização de campanhas, alocação de recursos, gestão de estoque, relacionamento com clientes e avaliação de desempenho comercial.